# 🚀 Aero Browse — Full Training Pipeline

**Vision-Language-Action (VLA) agent for continuous browser control**

This notebook runs the complete pipeline:
1. Install dependencies
2. Clone the Aero Browse repo
3. Fine-tune SmolVLM with LoRA on Multimodal-Mind2Web data
4. Merge LoRA adapter into base model
5. Convert merged model to GGUF (llama.cpp)
6. Push to Hugging Face Hub
7. Log results to Weights & Biases with live progress tracking

> **Runtime**: Kaggle GPU T4 x2 or P100 — ~2-4 hours for full pipeline

---

## 📋 0. Configuration

Set your secrets here. On Kaggle, use **Add-ons → Secrets** to store `HF_TOKEN` and `WANDB_API_KEY` securely.

In [ ]:
# ═══════════════════════════════════════════════════
# CONFIGURATION — Edit these before running
# ═══════════════════════════════════════════════════

# Model
BASE_MODEL_ID = "HuggingFaceTB/SmolVLM-Instruct"
REPO_URL = "https://github.com/HasinthakaPiyumal/aero-browse.git"

# Training hyperparameters
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4
MAX_STEPS = 500
LORA_R = 16
LORA_ALPHA = 32
MAX_SEQ_LENGTH = 1024
USE_SUBSET_SIZE = 1000  # Set to None for full training, or an integer to train on a smaller subset first

# Output paths
OUTPUT_DIR = "/kaggle/working/aero-browse-sft-output"
ADAPTER_DIR = "/kaggle/working/aero-browse_lora_adapter"
MERGED_DIR = "/kaggle/working/aero-browse-merged"
GGUF_DIR = "/kaggle/working/aero-browse-gguf"

# Hugging Face Hub
HF_REPO_ID = "YOUR_USERNAME/aero-browse-vla"  # ← Change this
HF_GGUF_REPO_ID = "YOUR_USERNAME/aero-browse-vla-gguf"  # ← Change this

# W&B
WANDB_PROJECT = "aero-browse"
WANDB_RUN_NAME = "smolvlm-lora-mind2web"

print("✅ Configuration loaded.")

## 🔑 1. Authenticate (Kaggle Secrets)

In [ ]:
import os

# Try loading from Kaggle secrets first, fall back to environment variables
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("✅ Loaded secrets from Kaggle vault.")
except Exception:
    print("⚠️  Kaggle secrets not available. Using environment variables.")
    print("   Set HF_TOKEN and WANDB_API_KEY manually if not already set.")

assert os.environ.get("HF_TOKEN"), "❌ HF_TOKEN not set! Add it to Kaggle Secrets."
assert os.environ.get("WANDB_API_KEY"), "❌ WANDB_API_KEY not set! Add it to Kaggle Secrets."
print("✅ Authentication ready.")

## 📦 2. Install Dependencies

In [ ]:
%%capture install_output
!pip install -q \
    torch \
    transformers \
    accelerate \
    peft \
    trl \
    bitsandbytes \
    datasets \
    wandb \
    huggingface_hub \
    sentencepiece \
    Pillow \
    numpy \
    opencv-python-headless \
    tqdm

print("✅ Core dependencies installed.")

In [ ]:
# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("❌ No GPU detected! Enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2")

## 📂 3. Clone Aero Browse Repository

In [ ]:
import subprocess, os

REPO_DIR = "/kaggle/working/aero-browse"

if os.path.exists(REPO_DIR):
    print(f"📁 Repo already exists at {REPO_DIR}, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print(f"📥 Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Add repo to Python path so we can import from src/
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"✅ Repo ready. Contents:")
for f in sorted(os.listdir(REPO_DIR)):
    print(f"   {'📁' if os.path.isdir(os.path.join(REPO_DIR, f)) else '📄'} {f}")

## 🧠 4. Initialize W&B Logging

In [ ]:
import wandb

wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "base_model": BASE_MODEL_ID,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRADIENT_ACCUMULATION,
        "learning_rate": LEARNING_RATE,
        "max_steps": MAX_STEPS,
        "max_seq_length": MAX_SEQ_LENGTH,
        "quantization": "4-bit NF4",
        "optimizer": "paged_adamw_8bit",
    },
    tags=["aero-browse", "smolvlm", "vla", "browser-agent"],
)
print(f"✅ W&B initialized: {wandb.run.url}")

## 🏋️ 5. Load Model, Tokenizer & Dataset — Then Train

In [ ]:
import torch
import numpy as np
from transformers import AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

try:
    from transformers import AutoModelForImageTextToText as AutoModelForVision2Seq
except ImportError:
    from transformers import AutoModelForVision2Seq

from src.vla_tokenizer import VLATokenizerConfig
from src.dataset import BrowserAgentDataset

# Enable progress bars for HF datasets
import datasets
datasets.logging.set_verbosity_info()

# ── Tokenizer + Custom Action Tokens ──
print("[1/5] Loading processor and injecting action tokens...")
vla_config = VLATokenizerConfig()
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
tokenizer = processor.tokenizer
num_added = tokenizer.add_tokens(vla_config.all_custom_tokens)
print(f"       Added {num_added} custom tokens (actions + coordinates).")

# ── 4-bit Quantized Model ──
print("[2/5] Loading base model in 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.resize_token_embeddings(len(tokenizer))

# ── LoRA Adapter ──
print("[3/5] Applying LoRA adapter...")
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    modules_to_save=["embed_tokens", "lm_head"],
    task_type="CAUSAL_LM",
    ensure_weight_tying=True,
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

print("✅ Model ready for training.")

In [ ]:
# ── Dataset ──
from datasets import load_dataset
import json

print("[4/5] Loading Multimodal-Mind2Web dataset from HF Hub...")
hf_dataset = load_dataset("osunlp/Multimodal-Mind2Web", split="train")

def has_bounding_box(example):
    try:
        candidates = example["pos_candidates"]
        if not candidates:
            return False
        for c_str in candidates:
            c = json.loads(c_str) if isinstance(c_str, str) else c_str
            attrs = c.get("attributes", {})
            if isinstance(attrs, str):
                attrs = json.loads(attrs)
            if attrs.get("bounding_box_rect"):
                return True
        return False
    except Exception:
        return False

print("       Filtering dataset for valid bounding boxes...")
# Use HF fast filtering (will show native tqdm progress bar)
filtered_dataset = hf_dataset.filter(has_bounding_box)
print(f"       Filtered dataset size: {len(filtered_dataset)} samples")

if USE_SUBSET_SIZE is not None and USE_SUBSET_SIZE < len(filtered_dataset):
    print(f"       Selecting a subset of {USE_SUBSET_SIZE} samples for training...")
    filtered_dataset = filtered_dataset.select(range(USE_SUBSET_SIZE))

train_dataset = BrowserAgentDataset(filtered_dataset, processor)

# Sanity check: inspect first sample
sample = train_dataset[0]
print(f"       Sample input_ids shape:             {sample['input_ids'].shape}")
print(f"       Sample pixel_values shape:           {sample['pixel_values'].shape}")
print(f"       Sample pixel_attention_mask shape:    {sample['pixel_attention_mask'].shape}")
print(f"       Sample labels shape:                {sample['labels'].shape}")
print(f"       Masked tokens (prompt):             {(sample['labels'] == -100).sum().item()}")
print(f"       Target tokens (action):             {(sample['labels'] != -100).sum().item()}")

In [ ]:
# ── Training ──
print("[5/5] Starting SFT training loop...")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    logging_steps=1,  # Log progress live on every single step!
    max_steps=MAX_STEPS,
    optim="paged_adamw_8bit",
    bf16=True,
    remove_unused_columns=False,
    save_strategy="steps",
    save_steps=100,
    max_length=MAX_SEQ_LENGTH,
    report_to="wandb",
    run_name=WANDB_RUN_NAME,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    disable_tqdm=True,  # Disable default progress bar to prevent double rendering
)

def collate_fn(examples):
    input_ids = [e["input_ids"] for e in examples]
    attention_mask = [e["attention_mask"] for e in examples]
    labels = [e["labels"] for e in examples]
    pixel_values = [e["pixel_values"] for e in examples]
    pixel_attention_mask = [e["pixel_attention_mask"] for e in examples]

    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

    # Pad pixel_values and pixel_attention_mask along patch dimension
    max_patches = max(p.shape[0] for p in pixel_values)
    padded_pixel_values = []
    padded_pixel_attention_mask = []

    for p, p_mask in zip(pixel_values, pixel_attention_mask):
        num_patches = p.shape[0]
        # Pad pixel_values: shape [num_patches, 3, 384, 384] -> [max_patches, 3, 384, 384]
        p_pad = torch.nn.functional.pad(p, (0, 0, 0, 0, 0, 0, 0, max_patches - num_patches), value=0)
        padded_pixel_values.append(p_pad)

        # Pad pixel_attention_mask: shape [num_patches, 384, 384] -> [max_patches, 384, 384]
        m_pad = torch.nn.functional.pad(p_mask, (0, 0, 0, 0, 0, max_patches - num_patches), value=0)
        padded_pixel_attention_mask.append(m_pad)

    pixel_values = torch.stack(padded_pixel_values)
    pixel_attention_mask = torch.stack(padded_pixel_attention_mask)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "pixel_values": pixel_values,
        "pixel_attention_mask": pixel_attention_mask
    }

# Custom TQDM Progress & W&B Live Callback
from transformers import TrainerCallback
from tqdm.notebook import tqdm

class TQDMProgressCallback(TrainerCallback):
    def __init__(self, max_steps):
        self.pbar = None
        self.max_steps = max_steps

    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=self.max_steps, desc="Fine-tuning VLA")

    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.n = state.global_step
            self.pbar.refresh()
        
        # Log progress live to W&B on every step
        progress_pct = (state.global_step / self.max_steps) * 100
        kwargs["trainer"].log({
            "progress_percent": progress_pct,
            "global_step_counter": state.global_step
        })

    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.close()

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    data_collator=collate_fn,
    callbacks=[TQDMProgressCallback(max_steps=MAX_STEPS)],
)

train_result = trainer.train()

# Save adapter
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Log final metrics
metrics = train_result.metrics
wandb.log({"final_train_loss": metrics.get("train_loss", None)})
print(f"\n✅ Training complete!")
print(f"   Final loss: {metrics.get('train_loss', 'N/A')}")
print(f"   Adapter saved to: {ADAPTER_DIR}")

## 🔗 6. Merge LoRA Weights into Base Model

In [ ]:
import torch, gc
from peft import PeftModel

try:
    from transformers import AutoModelForImageTextToText as AutoModelForVision2Seq
except ImportError:
    from transformers import AutoModelForVision2Seq

print("[Merge] Freeing training VRAM...")
del model, trainer
gc.collect()
torch.cuda.empty_cache()

print("[Merge] Loading base model in fp16 for merge...")
base_model = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",  # Merge on CPU to avoid OOM
)

# Resize embeddings to match the adapter (which had custom tokens)
from transformers import AutoTokenizer
merged_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base_model.resize_token_embeddings(len(merged_tokenizer))

print("[Merge] Loading LoRA adapter...")
model_with_lora = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

print("[Merge] Merging and unloading LoRA weights...")
merged_model = model_with_lora.merge_and_unload()

print(f"[Merge] Saving merged model to {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
merged_tokenizer.save_pretrained(MERGED_DIR)

wandb.log({"merge_status": "success"})
print(f"✅ Merged model saved to: {MERGED_DIR}")

## ⚙️ 7. Convert to GGUF (llama.cpp)

In [ ]:
import subprocess, os

LLAMA_CPP_DIR = "/kaggle/working/llama.cpp"

# Clone llama.cpp if not present
if not os.path.exists(LLAMA_CPP_DIR):
    print("[GGUF] Cloning llama.cpp...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git", LLAMA_CPP_DIR],
        check=True,
    )

# Install llama.cpp Python dependencies
print("[GGUF] Installing conversion dependencies...")
subprocess.run(
    ["pip", "install", "-q", "-r", os.path.join(LLAMA_CPP_DIR, "requirements.txt")],
    check=True,
)

print("✅ llama.cpp ready.")

In [ ]:
import subprocess, os

os.makedirs(GGUF_DIR, exist_ok=True)
CONVERT_SCRIPT = os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py")

# ── FP16 GGUF ──
gguf_fp16_path = os.path.join(GGUF_DIR, "aero-browse-f16.gguf")
print(f"[GGUF] Converting to FP16 GGUF...")
result = subprocess.run(
    ["python", CONVERT_SCRIPT, MERGED_DIR, "--outfile", gguf_fp16_path, "--outtype", "f16"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(f"⚠️  FP16 conversion output:\n{result.stderr}")
    print("   Note: SmolVLM is a vision-language model. If convert_hf_to_gguf.py doesn't")
    print("   support this architecture yet, the GGUF step may need a custom converter.")
    print("   The merged safetensors model is still fully usable for HF deployment.")
    wandb.log({"gguf_conversion": "failed", "gguf_error": result.stderr[:500]})
else:
    size_mb = os.path.getsize(gguf_fp16_path) / (1024 * 1024)
    print(f"✅ FP16 GGUF saved: {gguf_fp16_path} ({size_mb:.0f} MB)")
    wandb.log({"gguf_conversion": "success", "gguf_fp16_size_mb": size_mb})

# ── Q4_K_M Quantization (optional, smaller file) ──
QUANTIZE_BIN = os.path.join(LLAMA_CPP_DIR, "build", "bin", "llama-quantize")
gguf_q4_path = os.path.join(GGUF_DIR, "aero-browse-q4_k_m.gguf")

if os.path.exists(gguf_fp16_path) and os.path.getsize(gguf_fp16_path) > 0:
    # Build llama.cpp quantizer if needed
    if not os.path.exists(QUANTIZE_BIN):
        print("[GGUF] Building llama.cpp quantizer...")
        build_dir = os.path.join(LLAMA_CPP_DIR, "build")
        os.makedirs(build_dir, exist_ok=True)
        subprocess.run(["cmake", ".."], cwd=build_dir, check=True)
        subprocess.run(["cmake", "--build", ".", "--config", "Release", "-j"], cwd=build_dir, check=True)

    if os.path.exists(QUANTIZE_BIN):
        print("[GGUF] Quantizing to Q4_K_M...")
        subprocess.run([QUANTIZE_BIN, gguf_fp16_path, gguf_q4_path, "Q4_K_M"], check=True)
        q4_size = os.path.getsize(gguf_q4_path) / (1024 * 1024)
        print(f"✅ Q4_K_M GGUF saved: {gguf_q4_path} ({q4_size:.0f} MB)")
        wandb.log({"gguf_q4km_size_mb": q4_size})
    else:
        print("⚠️  llama-quantize binary not found, skipping Q4_K_M quantization.")
else:
    print("⚠️  Skipping Q4_K_M quantization (FP16 GGUF not available).")

## 📂 8. Push to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi, login

login(token=os.environ["HF_TOKEN"])
api = HfApi()

# ── Push merged safetensors model ──
print(f"[HF Hub] Uploading merged model to {HF_REPO_ID}...")
api.create_repo(HF_REPO_ID, exist_ok=True, private=False)
api.upload_folder(
    folder_path=MERGED_DIR,
    repo_id=HF_REPO_ID,
    commit_message="Upload Aero Browse VLA — merged LoRA + SmolVLM",
)
print(f"✅ Merged model uploaded: https://huggingface.co/{HF_REPO_ID}")

# ── Push GGUF files ──
gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf")]
if gguf_files:
    print(f"[HF Hub] Uploading GGUF files to {HF_GGUF_REPO_ID}...")
    api.create_repo(HF_GGUF_REPO_ID, exist_ok=True, private=False)
    api.upload_folder(
        folder_path=GGUF_DIR,
        repo_id=HF_GGUF_REPO_ID,
        commit_message="Upload Aero Browse VLA — GGUF quantized",
    )
    print(f"✅ GGUF models uploaded: https://huggingface.co/{HF_GGUF_REPO_ID}")
else:
    print("⚠️  No GGUF files to upload.")

# Also push the LoRA adapter separately (lightweight, useful for others)
ADAPTER_REPO_ID = HF_REPO_ID + "-lora"
print(f"[HF Hub] Uploading LoRA adapter to {ADAPTER_REPO_ID}...")
api.create_repo(ADAPTER_REPO_ID, exist_ok=True, private=False)
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=ADAPTER_REPO_ID,
    commit_message="Upload Aero Browse VLA — LoRA adapter only",
)
print(f"✅ LoRA adapter uploaded: https://huggingface.co/{ADAPTER_REPO_ID}")

wandb.log({"hf_push_status": "success", "hf_repo": HF_REPO_ID})

## 📊 9. Final W&B Summary & Cleanup

In [ ]:
import os

# Log artifact sizes
def get_dir_size(path):
    total = 0
    if os.path.exists(path):
        for dirpath, dirnames, filenames in os.walk(path):
            for f in filenames:
                total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 * 1024)  # MB

summary = {
    "base_model": BASE_MODEL_ID,
    "custom_tokens_added": num_added,
    "training_samples": len(train_dataset),
    "max_steps": MAX_STEPS,
    "adapter_size_mb": get_dir_size(ADAPTER_DIR),
    "merged_model_size_mb": get_dir_size(MERGED_DIR),
    "gguf_dir_size_mb": get_dir_size(GGUF_DIR),
    "hf_repo": HF_REPO_ID,
}

wandb.log(summary)
wandb.finish()

print("\n" + "="*60)
print("  🎉 AERO BROWSE PIPELINE COMPLETE")
print("="*60)
print(f"  📦 LoRA Adapter:    {ADAPTER_DIR}")
print(f"  🔗 Merged Model:    {MERGED_DIR}")
print(f"  ⚙️  GGUF Models:     {GGUF_DIR}")
print(f"  🤗 HF Hub:          https://huggingface.co/{HF_REPO_ID}")
print(f"  📊 W&B Dashboard:   Check your wandb.ai/{WANDB_PROJECT}")
print("="*60)